# GI Vertex Group Remap Finder
[![Static Badge](https://img.shields.io/badge/Jupyter_Notebook-F37726?style=for-the-badge)](https://jupyter.org/)

<br>

Proposes the vertex group remap between two character skins for the game GI, from the geometry of both characters (their 3dmigoto dumps, or a mod's raw `.buf` files), and writes it as a draft workbook in the format of [`Data/RemapDrafts`](../../../Data/RemapDrafts/README.md)

<br>

## Contributors

|   |   |
|---|---|
| **[Albert Gold](https://github.com/Alex-Au1)** | [![](https://dcbadge.limes.pink/api/shield/367087171154214914?theme=discord-inverted)](https://discordlookup.com/user/367087171154214914) |

<br>

## Requirements
- Python (Version 3.6 or up)
- [numpy](https://numpy.org/), [openpyxl](https://openpyxl.readthedocs.io/) (for the workbook), [pandas](https://pandas.pydata.org/) (only to display the tables in this notebook) and [scipy](https://scipy.org/) (speeds up the nearest-vertex mode; the finder falls back to a slower search without it) --- all listed in [`requirements.txt`](../requirements.txt)

<br>
<br>

## Installation
First install the libraries the finder itself needs, from the [`requirements.txt`](../requirements.txt) next to this notebook's folder


In [ ]:
%pip install -r ../requirements.txt

<br>

Then choose how to install AGRemap's API

**Option A**: If you want to install through [Pypi](https://pypi.org/project/AnimeGameRemap/), you run the pip install command below


In [ ]:
%pip install -U AnimeGameRemap

In [ ]:
import AnimeGameRemap as AGR

<br>

**Option B**: Alternatively, you can locally import the API from a specific git branch


In [ ]:
import os
import sys

# Note: Make sure the path correctly points where the AGRemap's API is located
#   (it must end up absolute: the API's native extensions cannot be loaded from a relative sys.path entry)
sys.path.insert(1, os.path.abspath(r"../../../Anime Game Remap (for all users)/api/src/py"))

import FixRaidenBoss2 as AGR

<br>
<br>

## How is the Remap Found?

> ***📝 NOTE:*** <br>
>
> If you do not care about how the proposal is worked out, you can skip this section and proceed to the [Initialization](#initialization) codeblock
>

<br>

### Overview

Every vertex of a GI character is *skinned* to up to 4 bones of the character's skeleton: its **Blend.buf** line holds 4 bone indices (**BlendIndices**) and how strongly each bone pulls on the vertex (**BlendWeight**).
The set of vertices pulled by one bone is that bone's **vertex group**, and a bone is only ever referred to by its index.

Two skins of the same character (*Ganyu* and *GanyuTwilight*, say) have **different skeletons with different numbering**, so a mod made for one skin drives the wrong bones on the other.
A **vertex group remap** is the table that fixes this --- for every bone index of the skin the mod was made for, the index of the corresponding bone on the skin it is being remapped onto.
This is the table that [`VGRemapData`](../../../Anime%20Game%20Remap%20(for%20all%20users)/api/src/cpp/core/src/data/VGRemapData.cpp) ships for every supported pair of skins, and that used to be worked out by hand in [`Data/RemapDrafts`](../../../Data/RemapDrafts/README.md).

The finder proposes that table from the geometry alone, in 4 steps:

| Step | What it does |
| ---- | ------------ |
| **1. Read** | Load every vertex's position and its bone indices/weights, for both skins |
| **2. Summarise** | Boil each vertex group down to *where* it is (its centre) and *how far it reaches* (its spread) |
| **3. Compare** | Measure how far apart every vertex group of one skin is from every vertex group of the other |
| **4. Match** | Pick, for each vertex group of the skin being remapped, the vertex group of the other skin it corresponds to --- aligning whole chains of bones at once rather than one bone at a time |

<br>

### Step 1: Reading the geometry

Only two things about a vertex matter here: where it is, and which bones pull on it. Both forms of a character's geometry carry them:

| Form | Positions | Bone indices and weights | Which object a vertex is drawn in |
| ---- | --------- | ------------------------ | --------------------------------- |
| **3dmigoto dumps** (a folder of [GI-Model-Importer-Assets](https://github.com/SilentNightSound/GI-Model-Importer-Assets)' `PlayerCharacterData`) | the `POSITION` element of the `*-vb0=<hash>.txt` | the `BLENDINDICES` / `BLENDWEIGHT` elements of the same file | the `*-ib=<hash>.txt` files (one per object) |
| **A mod's raw files** | `*Position.buf` | `*Blend.buf` | the `*.ib` files (one per object) |

The files are read through the API's `VbFile`, `PositionFile`, `BlendFile` and `IbFile`, so both forms decode to exactly the same numbers (see the [Mod to Dump Converter](../../ModToDumpConverter/GI/GIModToDumpConverter.ipynb) for the byte-level layout of these files).
The `Texcoord.buf` and the textures are not needed.
Which object (`Head`, `Body`, `Dress`, ...) a vertex is drawn in is only recorded for the comments; it plays no part in the matching.

A vertex belongs to a vertex group when it carries that group's index with a **non-zero weight**. The number of vertex groups a skin has is one past the largest index in use --- a bone no vertex is skinned to still exists, so the remap still needs a row for it.

<br>

### Step 2: Summarising a vertex group

For a vertex group $V$ with vertices at positions $p_1, \dots, p_n$ carrying weights $w_1, \dots, w_n$ for that bone, the finder keeps two things:

| Summary | Formula | Meaning |
| ------- | ------- | ------- |
| **Centre** $\mu$ | $\mu = \dfrac{\sum_i w_i \, p_i}{\sum_i w_i}$ | Where the group sits |
| **Spread** $\Sigma$ | $\Sigma = \dfrac{\sum_i w_i \, (p_i - \mu)(p_i - \mu)^T}{\sum_i w_i}$ | How far the group reaches, and in which directions (a $3 \times 3$ covariance) |

Weighting by $w_i$ means a vertex a bone barely pulls on barely moves that bone's summary. This looked pointless on a single character and turned out to be worth several points of accuracy over all of them (see the table at the end); setting `Weighted = False` below counts every vertex equally instead.

> ***📝 NOTE:*** <br>
>
> A position in these files is stored as $(x, z, y)$ rather than $(x, y, z)$. It makes no difference here: a distance between two points is the same whichever way the axes are labelled, as long as both skins agree
>

<br>

### Step 3: Comparing two vertex groups

With a centre and a spread, each vertex group is treated as a Gaussian blob $\mathcal{N}(\mu, \Sigma)$, and the distance between a group $A$ of one skin and a group $B$ of the other is the [2-Wasserstein distance](https://en.wikipedia.org/wiki/Wasserstein_metric#Normal_distributions) between the two blobs:

$$ W_2(A, B)^2 = \lVert \mu_A - \mu_B \rVert^2 + \operatorname{tr}\left( \Sigma_A + \Sigma_B - 2 \left( \Sigma_B^{1/2} \Sigma_A \Sigma_B^{1/2} \right)^{1/2} \right) $$

The first term is just the distance between the centres; the second is how differently the two groups are shaped, in the *same units*, so the two combine with nothing to tune. Two groups whose centres coincide but where one is a long strand and the other a small knot are still far apart.
(`Metric = VGMatcher.MetricCenter` keeps only the first term.)

<br>

### Step 4: Matching --- whole chains at once

The simplest match sends every vertex group of the skin being remapped to the *closest* group of the other skin (`Mode = VGMatcher.ModeNearest`). It is right about 4 times in 5, and nearly all of its mistakes look the same: **neighbouring bones along a chain** --- a strand of hair, a ribbon, the rows of a skirt --- whose centres sit a few millimetres apart, so a small difference in the two skins' proportions flips which one is nearest, and a whole run of bones lands one bone off.

The fix uses one fact about how the game numbers its bones: a chain of bones gets **consecutive indices, on both skins**. So instead of choosing each match on its own, the finder aligns every run of consecutive source indices onto the target indices *as a whole* (`Mode = VGMatcher.ModeChains`), choosing the alignment with the lowest total cost, where the cost of one bone's match is its distance from step 3, plus a cost for how the target index moved since the previous bone:

| The target index... | Cost | Why |
| ------------------- | ---- | --- |
| went up by exactly 1 | free | The two chains run in step |
| went up by 2 | `SkipCost` $\times$ spacing | The other skin has one extra bone in this chain |
| stayed the same | `StayCost` $\times$ spacing | Two bones of this skin share one bone on the other (allowed --- several rows of the remap may hold the same value --- but not for free) |
| anything else | `JumpCost` $\times$ spacing | A new chain starts here |

"Spacing" is the typical distance between a bone and its nearest neighbour on the target skin, so the same costs mean the same thing on a small character and a large one. The lowest-cost alignment of a whole run is found exactly by dynamic programming (the same [Viterbi](https://en.wikipedia.org/wiki/Viterbi_algorithm) recurrence used for sequence alignment), one run at a time.

**Uncertainty** (column C of the draft) comes out of the same computation: for each bone, how much *more* the whole run's best alignment would cost if that one bone were forced onto its runner-up. A large margin means the choice was clear (uncertainty near 0); no margin means a coin toss (uncertainty near 1). The **Comments** column carries the runner-up, the distances, the objects each group is drawn in, and the chain the match is part of, so a doubtful row can be checked without re-running anything.

<br>

### How good is it?

Scored against the hand-made drafts in `Data/RemapDrafts` (every direction whose two skins have dumps available: 20 directions over 11 characters, 1993 vertex groups, as of 2026-09-09), by the tool's `benchmark.py`:

| | `ModeNearest` | `ModeChains` |
| --- | --- | --- |
| `MetricCenter`, unweighted | 81.9% | 85.5% |
| `MetricGaussian`, unweighted | 82.9% | 87.1% |
| `MetricCenter`, weighted | 87.1% | 89.1% |
| `MetricGaussian`, weighted (**default**) | 87.8% | **89.6%** |

What is still wrong is mostly of two kinds, and both come out with a high uncertainty, so sort the draft by column C and check those rows first:

- A hand-made draft sometimes sends a bone the other skin simply does not have to bone `0`. No geometry can predict that.
- Where the two skins' proportions differ, the true counterpart of a chain's *root* is sometimes not its nearest bone (Ganyu's hair root `16` maps to GanyuTwilight's `4`, though `5` is much closer).

> ***📝 NOTE:*** <br>
>
> The proposal is a **draft**, not an answer. Check it in game the same way a hand-made one is checked, starting with the rows of highest uncertainty
>


<br>
<br>

## Initialization
Run the codeblock below to initialize the finder's tools (they live in the [`src/VGRemapFinder`](../src/VGRemapFinder) package next to this notebook, and read the geometry through the API's `VbFile`/`PositionFile`/`BlendFile`/`IbFile`)


In [ ]:
import os
import sys

# Note: Make sure the path correctly points to this tool's folder (the one holding 'src')
sys.path.insert(1, os.path.abspath(r".."))

from src.VGRemapFinder.DumpMod import DumpMod, setAPI
from src.VGRemapFinder.VertexGroups import VertexGroups
from src.VGRemapFinder.VGMatcher import VGMatcher
from src.VGRemapFinder.DraftWriter import DraftWriter
from src.VGRemapFinder.VGRemapFinder import VGRemapFinder

# use the API imported above, whichever option was chosen
setAPI(AGR)

<br>
<br>

## File Setup
Ensure the paths are set correctly for the following constants:

- **FromFolder**: the geometry of the character to be remapped
- **ToFolder**: the geometry of the character to remap onto
- **OutputPath**: where to write the draft workbook (`None` to not write one). The workbook carries an `About` sheet marking it as a proposal; the tool's `benchmark.py` skips such workbooks, so remove that sheet once the proposal has been checked and is a draft in its own right
- **CompareTo**: an existing draft workbook to score the proposal against (`None` to skip)

Each folder is one of: a character's dump folder, in the layout of [GI-Model-Importer-Assets](https://github.com/SilentNightSound/GI-Model-Importer-Assets)' `PlayerCharacterData` (`*-vb0=<hash>.txt` and `*-ib=<hash>.txt` files); the folder of a mod's raw files (`*Position.buf`, `*Blend.buf` and the `*.ib` files); or a raw 3dmigoto frame analysis straight out of the game, in which case the character's **position, blend and ib hashes** (**FromHashes** / **ToHashes**, the `position_vb` / `blend_vb` / `ib` of a `hash.json`) pick its files out of the thousands there --- run with them left as `None` to be shown which hashes the folder holds. Which form a folder is is worked out from what it holds. Subfolders are not searched, so point at the folder that holds the files

- **CompareLibrary**: whether to also score the proposal against the remap the AG Remap library ships for the pair (the names must be the library's own, eg. `Keqing` / `KeqingOpulent`)


In [ ]:
#####################
# Ensure the paths set in these constants are correct

FromFolder = r"../../../../GI-Model-Importer-Assets/PlayerCharacterData/Ganyu"
ToFolder = r"../../../../GI-Model-Importer-Assets/PlayerCharacterData/GanyuTwilight"
FromHashes = None
ToHashes = None

# eg. from a mod's raw .buf files instead:
# FromFolder = r"../../../Data/Mod Downloads/GI/Ganyu/4_0"

# eg. straight from raw frame analyses (position, blend, ib hashes):
# FromFolder = r"E:/Computer/Games/Wuthering Waves Mods/Importer/GIMI/FrameAnalysis-KeqingOpulent 2026-09-09-213520"
# FromHashes = ("0d7e3cc5", "6f010b58", "7c6fc8c3")
# ToFolder = r"E:/Computer/Games/Wuthering Waves Mods/Importer/GIMI/FrameAnalysis-Keqing-2026-09-09-213224"
# ToHashes = ("3aaf3e94", "0bf8e621", "cbf1894b")

OutputPath = r"../../../Data/RemapDrafts/GanyuRemapDraft (proposed).xlsx"
CompareTo = r"../../../Data/RemapDrafts/GanyuRemapDraft.xlsx"
CompareLibrary = False

#####################

# The names heading the workbook's columns (None: taken from the files) and the game version, for the sheet titles
FromName = None
ToName = None
Version = "4.4"

# Whether to also propose the reverse remap (ToFolder onto FromFolder), as a second sheet
BothWays = True

<br>

### Options
How the vertex groups are compared and matched --- see [How is the Remap Found?](#how-is-the-remap-found) above:

- **Metric**: `VGMatcher.MetricGaussian` compares each group's centre *and* the spread of its vertices, `VGMatcher.MetricCenter` the centres only
- **Mode**: `VGMatcher.ModeChains` aligns each run of consecutive vertex group indices onto the other skin's indices as a whole chain, `VGMatcher.ModeNearest` maps every group independently onto its closest one, and `VGMatcher.ModeVertices` ignores the summaries and tallies which bones drive the *nearest skin* on the other skin for every vertex a group drives --- worse as a default (83% on the benchmark) but the right tool for a part the other skin does not have, since it says what moves the skin underneath it. Read its share and runner-up in the comments, and overrule it when the winner is hair and the part is not
- **StayCost**, **SkipCost**, **JumpCost**: the chain alignment's step costs, in units of the target skin's typical bone spacing
- **Weighted**: whether each vertex counts by its blend weight when summarising a group, rather than every vertex counting equally


In [ ]:
Metric = VGMatcher.MetricGaussian
Mode = VGMatcher.ModeChains

StayCost = VGMatcher.DefaultStayCost
SkipCost = VGMatcher.DefaultSkipCost
JumpCost = VGMatcher.DefaultJumpCost

Weighted = True

<br>
<br>

## Run the Finder
The code block below reads both characters' geometry, proposes the remap(s), prints a summary of each (and the agreement with **CompareTo**, if set), and writes the workbook to **OutputPath**


In [ ]:
finder = VGRemapFinder(FromFolder, ToFolder, fromName = FromName, toName = ToName, version = Version,
                       weighted = Weighted, bothWays = BothWays, metric = Metric, mode = Mode,
                       stayCost = StayCost, skipCost = SkipCost, jumpCost = JumpCost,
                       fromHashes = FromHashes, toHashes = ToHashes)

sheets = finder.run(output = OutputPath, compareTo = CompareTo, compareLibrary = CompareLibrary)

<br>
<br>

## Inspect the Proposal
The code block below shows each proposed remap as a table, in the same columns as the workbook. Sort by **Uncertainty** to see which rows to check first


In [ ]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

for fromName, toName, version, matches in sheets:
    table = pd.DataFrame({fromName: [match.fromIndex for match in matches],
                          toName: [match.toIndex for match in matches],
                          "Uncertainty": [round(match.uncertainty, 3) for match in matches],
                          "Comments": [match.comment() for match in matches]})

    print(f"{fromName} -> {toName}")
    display(table.style.hide(axis = "index"))